# Project 2 — Environment and Cloud setup

Run this notebook on the Mac's Jupyter kernel for local preflight and Google Cloud setup. It is the only setup entry point. Spark/data analysis runs in `01_data_quality.ipynb` on the Dataproc Jupyter kernel.

The only permitted billing account is Education (ID `01DAD8-AE5F1A-E3910A`); the only permitted credit is `3LU63BPGY77JFDQQ`. The credit was directly verified on the Education Billing Console Credits page on 2026-10-11; only one issued credit was present, Available with USD 50 remaining. This notebook creates and links a dedicated project to Education because the user explicitly requested that account. It does not create paid resources by default.

In [1]:
from pathlib import Path
import json
import os
import subprocess

from IPython.display import display

PROJECT_ID = "sg-rainfall-dsa5208-261011"
BILLING_ACCOUNT_NAME = "Billing Account for Education"
BILLING_ACCOUNT_ID = "01DAD8-AE5F1A-E3910A"
ALLOWED_CREDIT_ID = "3LU63BPGY77JFDQQ"
REGION = "asia-southeast1"
CLUSTER = "dsa5208-rainfall"
BUCKET = "sg-rainfall-dsa5208-261011-data"  # Fill only after checking global availability and estimating storage cost.
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "../rainfall-data")).expanduser()
def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "scripts").is_dir() and (candidate / "manifests").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from this notebook.")

REPO_ROOT = find_repo_root()

CREDIT_SNAPSHOT = {
    "credit_id": ALLOWED_CREDIT_ID,
    "status": "Available",
    "percent_remaining": 100,
    "remaining_usd": 50.00,
    "original_usd": 50.00,
    "snapshot_source": "Education Billing Console, directly verified",
    "snapshot_date": "2026-10-11",
}
COST_RESERVE_USD = 10.00
MAX_PLANNED_SPEND_USD = 40.00
CREDIT_LIVE_VERIFIED = True  # Refresh from the Education account Credits page first.
PLANNED_SESSION_COST_USD = 8.0  # Use an estimate including pending and cleanup costs.

print(f"Notebook root: {REPO_ROOT}")
print(f"Project: {PROJECT_ID}")
print(f"Target billing: {BILLING_ACCOUNT_NAME} ({BILLING_ACCOUNT_ID})")
print(f"Allowed credit: {ALLOWED_CREDIT_ID}")
print("Credit directly verified on Education Credits page on 2026-10-11: 50 USD available.")

Notebook root: /Users/macbook/Desktop/workspace/sg-rainfall-distributed-analysis-and-prediction
Project: sg-rainfall-dsa5208-261011
Target billing: Billing Account for Education (01DAD8-AE5F1A-E3910A)
Allowed credit: 3LU63BPGY77JFDQQ
Credit directly verified on Education Credits page on 2026-10-11: 50 USD available.


In [2]:
REBUILD_MANIFEST = False  # Set True to rescan the eight local annual inputs.
manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
if REBUILD_MANIFEST:
    import importlib.util

    module_path = REPO_ROOT / "scripts" / "build_manifest.py"
    spec = importlib.util.spec_from_file_location("build_manifest", module_path)
    manifest_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(manifest_module)
    rebuilt = manifest_module.build_manifest(DATA_ROOT)
    temporary_path = manifest_path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(rebuilt, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(manifest_path)
    print(f"Rebuilt manifest: {manifest_path}")

if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_rows = [
    {
        "year": item["year"],
        "rows": item["row_count"],
        "size_bytes": item["size_bytes"],
        "sha256": item["sha256"],
        "schema_matches": item["schema_matches_expected"],
        "malformed_rows": item["rows_with_wrong_field_count"],
        "invalid_timestamps": item["unparseable_or_missing_timestamps"],
    }
    for item in manifest["files"]
]
display(manifest_rows)
print(f"Input years: {manifest['selection']['years']}")
print(f"Files in manifest: {len(manifest_rows)}")
print("Raw CSV files are read-only.")

[{'year': 2017,
  'rows': 5256106,
  'size_bytes': 881206891,
  'sha256': 'a51a27378607c6c37d51b72383259b27dae5dc7d5760a8b8fff04e86d9f92ddf',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2018,
  'rows': 4594246,
  'size_bytes': 769547920,
  'sha256': 'f42b40c37edbbc9a3ddb6e53e147606878280cbe1362e6580ea86aaf528db887',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2019,
  'rows': 5026862,
  'size_bytes': 842751786,
  'sha256': '1f30310b94912c800a96a9beedd65fcff62c663a1cbc2ff5606f0f174867d73c',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2020,
  'rows': 6334473,
  'size_bytes': 1048946924,
  'sha256': '2b8df4c6adebb3ea41183ca22fc261fdb34ef78a492bb205f660c431b924ec48',
  'schema_matches': True,
  'malformed_rows': 0,
  'invalid_timestamps': 0},
 {'year': 2021,
  'rows': 7044646,
  'size_bytes': 1158573170,
  'sha256': '3bb97164ec298b4057303bfecea4acc0a824fc41d83aa3d

Input years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Files in manifest: 8
Raw CSV files are read-only.


In [3]:
CREATE_PROJECT = True

if CREATE_PROJECT:
    result = subprocess.run(
        ["gcloud", "projects", "describe", PROJECT_ID, "--format=json"],
        capture_output=True, text=True,
    )
    if result.returncode:
        created = subprocess.run(
            ["gcloud", "projects", "create", PROJECT_ID, "--name=DSA5208 Rainfall"],
            capture_output=True, text=True,
        )
        if created.returncode:
            raise RuntimeError("Project creation failed. Check project-creation permissions in Console.")
        print(f"Created project: {PROJECT_ID}")
    else:
        print(f"Project already exists: {PROJECT_ID}")

Project already exists: sg-rainfall-dsa5208-261011


In [4]:
import re

def run_gcloud(args, *, json_output=False):
    result = subprocess.run(
        ["gcloud", *args, *([f"--billing-project={PROJECT_ID}"] if args[:2] == ["billing", "budgets"] else [])],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        message = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr.strip())
        raise RuntimeError(message or f"gcloud exited with {result.returncode}")
    if json_output:
        return json.loads(result.stdout or "null")
    if result.stdout.strip():
        print(result.stdout.strip())
    return result.stdout

def billing_project_state():
    return run_gcloud(
        ["billing", "projects", "describe", PROJECT_ID, "--format=json"],
        json_output=True,
    )

before = billing_project_state()
target = f"billingAccounts/{BILLING_ACCOUNT_ID}"
print({
    "project_id": PROJECT_ID,
    "billing_enabled_before": before.get("billingEnabled"),
    "already_linked_to_education": before.get("billingAccountName") == target,
})

{'project_id': 'sg-rainfall-dsa5208-261011', 'billing_enabled_before': True, 'already_linked_to_education': True}


In [5]:
LINK_PROJECT_TO_EDUCATION = True

if LINK_PROJECT_TO_EDUCATION:
    current = billing_project_state()
    if current.get("billingAccountName") == target and current.get("billingEnabled") is True:
        verified = current
        billing_ready = True
    else:
        try:
            run_gcloud([
                "billing", "projects", "link", PROJECT_ID,
                f"--billing-account={BILLING_ACCOUNT_ID}",
            ])
            verified = billing_project_state()
            billing_ready = (
                verified.get("billingAccountName") == target
                and verified.get("billingEnabled") is True
            )
        except RuntimeError as exc:
            message = str(exc)
            if "IAM_PERMISSION_DENIED" in message or "does not have permission" in message:
                print(
                    "Billing link blocked: the active gcloud identity lacks "
                    "project billing-assignment permissions. No bucket, cluster, or job was created."
                )
                verified = current
                billing_ready = False
            else:
                raise

    display({
        "project_id": PROJECT_ID,
        "billing_account": BILLING_ACCOUNT_NAME,
        "billing_account_id": BILLING_ACCOUNT_ID,
        "billing_enabled": verified.get("billingEnabled"),
        "linkage_verified": verified.get("billingAccountName") == target,
    })
else:
    billing_ready = False
    print("Skipped billing linkage. No cloud resources may be created.")

{'project_id': 'sg-rainfall-dsa5208-261011',
 'billing_account': 'Billing Account for Education',
 'billing_account_id': '01DAD8-AE5F1A-E3910A',
 'billing_enabled': True,
 'linkage_verified': True}

In [6]:
credit_id_matches = CREDIT_SNAPSHOT["credit_id"] == ALLOWED_CREDIT_ID
credit_snapshot_available = (
    CREDIT_SNAPSHOT["status"] == "Available"
    and CREDIT_SNAPSHOT["remaining_usd"] > COST_RESERVE_USD
)

def require_cost_gate(estimate_usd):
    if not billing_ready:
        raise RuntimeError("Project is not linked to the permitted Education account.")
    if not CREDIT_LIVE_VERIFIED:
        raise RuntimeError("Refresh and verify this exact credit in Billing Console before paid work.")
    if not credit_id_matches or not credit_snapshot_available:
        raise RuntimeError("The sole permitted credit is missing, unavailable, or below its reserve.")
    if estimate_usd is None or estimate_usd < 0:
        raise RuntimeError("Enter a documented cost estimate before paid work.")
    allowed = min(MAX_PLANNED_SPEND_USD, CREDIT_SNAPSHOT["remaining_usd"] - COST_RESERVE_USD)
    if estimate_usd > allowed:
        raise RuntimeError(f"Estimate exceeds the allowed credit after reserve ({allowed:.2f} USD).")

display({
    "allowed_credit_id": ALLOWED_CREDIT_ID,
    "snapshot_credit_id_matches": credit_id_matches,
    "snapshot_remaining_usd": CREDIT_SNAPSHOT["remaining_usd"],
    "snapshot_is_live": CREDIT_LIVE_VERIFIED,
    "billing_project_ready": billing_ready,
    "paid_resource_gate": False,
})
print("No paid resource is permitted until the live credit snapshot and documented run estimate pass require_cost_gate(...).")

{'allowed_credit_id': '3LU63BPGY77JFDQQ',
 'snapshot_credit_id_matches': True,
 'snapshot_remaining_usd': 50.0,
 'snapshot_is_live': True,
 'billing_project_ready': True,
 'paid_resource_gate': False}

No paid resource is permitted until the live credit snapshot and documented run estimate pass require_cost_gate(...).


In [7]:
ENABLE_APIS = True
REQUIRED_APIS = [
    "cloudresourcemanager.googleapis.com",
    "serviceusage.googleapis.com",
    "iap.googleapis.com",
    "dataproc.googleapis.com",
    "compute.googleapis.com",
    "storage.googleapis.com",
    "iam.googleapis.com",
    "cloudbilling.googleapis.com",
    "billingbudgets.googleapis.com",
]

if ENABLE_APIS:
    if not billing_ready:
        raise RuntimeError("Billing preflight must pass before enabling APIs.")
    for api in REQUIRED_APIS:
        run_gcloud([
            "services", "enable", api,
            f"--project={PROJECT_ID}",
        ])
else:
    print("API enablement skipped; set ENABLE_APIS=True only after live credit and cost preflight.")

In [8]:
# Read-only regional quota and official list-price lookup.
import urllib.request
import urllib.parse

region_info = run_gcloud(["compute", "regions", "describe", REGION, f"--project={PROJECT_ID}", "--format=json"], json_output=True)
display([q for q in region_info.get("quotas", []) if q["metric"] in ("CPUS", "E2_CPUS", "DISKS_TOTAL_GB", "SSD_TOTAL_GB", "IN_USE_ADDRESSES")])

def public_skus(service):
    token = subprocess.run(["gcloud", "auth", "print-access-token"], capture_output=True, text=True, check=True).stdout.strip()
    page = ""
    found = []
    while True:
        query = urllib.parse.urlencode({"currencyCode": "USD", "pageSize": 5000, "pageToken": page})
        request = urllib.request.Request(f"https://cloudbilling.googleapis.com/v1/services/{service}/skus?{query}", headers={"Authorization": f"Bearer {token}", "x-goog-user-project": PROJECT_ID})
        with urllib.request.urlopen(request, timeout=60) as response:
            data = json.load(response)
        for sku in data.get("skus", []):
            description = sku["description"]
            if REGION in sku.get("serviceRegions", []) and sku.get("category", {}).get("usageType") == "OnDemand" and (description.startswith("E2 Instance") or "Balanced PD Capacity" in description):
                found.append(sku)
        page = data.get("nextPageToken", "")
        if not page:
            break
    return found

price_skus = public_skus("6F81-5844-456A")
price_rows = []
for sku in price_skus:
    expression = sku["pricingInfo"][-1]["pricingExpression"]
    rates = [float(r["unitPrice"].get("units", 0)) + r["unitPrice"].get("nanos", 0) / 1e9 for r in expression["tieredRates"]]
    price_rows.append({"sku": sku["skuId"], "description": sku["description"], "unit": expression["usageUnit"], "rates_usd": rates})
display(price_rows)


[{'limit': 100.0, 'metric': 'CPUS', 'usage': 0.0},
 {'limit': 2048.0, 'metric': 'DISKS_TOTAL_GB', 'usage': 0.0},
 {'limit': 8.0, 'metric': 'IN_USE_ADDRESSES', 'usage': 0.0},
 {'limit': 250.0, 'metric': 'SSD_TOTAL_GB', 'usage': 0.0},
 {'limit': 24.0, 'metric': 'E2_CPUS', 'usage': 0.0}]

[{'sku': '16D2-FC49-A127',
  'description': 'Balanced PD Capacity in Singapore',
  'unit': 'GiBy.mo',
  'rates_usd': [0.11]},
 {'sku': '7102-B9AA-2ACF',
  'description': 'E2 Instance Core running in Singapore',
  'unit': 'h',
  'rates_usd': [0.02690931]},
 {'sku': 'BB09-D9FE-73FD',
  'description': 'Regional Balanced PD Capacity in Singapore',
  'unit': 'GiBy.mo',
  'rates_usd': [0.22]},
 {'sku': 'E277-7DC2-2C7D',
  'description': 'E2 Instance Ram running in Singapore',
  'unit': 'GiBy.h',
  'rates_usd': [0.00360594]}]

In [9]:
# Three-hour session, based on official Singapore SKU rates above.
core_rate = next(r["rates_usd"][0] for r in price_rows if r["description"].startswith("E2 Instance Core"))
ram_rate = next(r["rates_usd"][0] for r in price_rows if r["description"].startswith("E2 Instance Ram"))
disk_rate = next(r["rates_usd"][0] for r in price_rows if r["description"] == "Balanced PD Capacity in Singapore")
compute_estimate = 3 * 3 * (4 * core_rate + 16 * ram_rate)
management_estimate = 12 * 3 * 0.01
# Disk prorated using a conservative 28-day month; 4 USD allowance covers
# GCS operations/storage, result export/download, logging and cleanup.
disk_estimate = 250 * disk_rate * 3 / (28 * 24)
estimate_with_allowance = compute_estimate + management_estimate + disk_estimate + 4.0
if estimate_with_allowance > PLANNED_SESSION_COST_USD:
    raise RuntimeError("Session estimate exceeds the reserved eight-dollar allocation.")
require_cost_gate(PLANNED_SESSION_COST_USD)
display({"compute_usd": compute_estimate, "dataproc_management_usd": management_estimate, "disk_usd": disk_estimate, "other_allowance_usd": 4.0, "estimated_usd": estimate_with_allowance, "reserved_session_usd": PLANNED_SESSION_COST_USD})
project_info = run_gcloud(["projects", "describe", PROJECT_ID, "--format=json"], json_output=True)
PROJECT_NUMBER = project_info["projectNumber"]
budgets = run_gcloud(["billing", "budgets", "list", f"--billing-account={BILLING_ACCOUNT_ID}", "--format=json"], json_output=True)
if not any(item.get("displayName") == "DSA5208 Rainfall 40 USD" for item in budgets):
    run_gcloud(["billing", "budgets", "create", f"--billing-account={BILLING_ACCOUNT_ID}", "--display-name=DSA5208 Rainfall 40 USD", "--budget-amount=40USD", f"--filter-projects=projects/{PROJECT_NUMBER}", "--credit-types-treatment=exclude-all-credits", "--threshold-rule=percent=0.5", "--threshold-rule=percent=0.75", "--threshold-rule=percent=0.9", "--start-date=2026-10-11", "--end-date=2026-10-19"])
print("Budget configured for project gross usage (alerts only; not an automatic spending cap).")

{'compute_usd': 1.4879905199999999,
 'dataproc_management_usd': 0.36,
 'disk_usd': 0.12276785714285714,
 'other_allowance_usd': 4.0,
 'estimated_usd': 5.970758377142857,
 'reserved_session_usd': 8.0}

Budget configured for project gross usage (alerts only; not an automatic spending cap).


In [10]:
NETWORK = "rainfall-private"
SUBNET_NAME = "rainfall-singapore"
VM_SERVICE_ACCOUNT = f"rainfall-vm@{PROJECT_ID}.iam.gserviceaccount.com"

def resource_exists(args):
    return subprocess.run(["gcloud", *args, "--format=json"], capture_output=True, text=True).returncode == 0

if not resource_exists(["compute", "networks", "describe", NETWORK, f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "networks", "create", NETWORK, "--subnet-mode=custom", f"--project={PROJECT_ID}"])
if not resource_exists(["compute", "networks", "subnets", "describe", SUBNET_NAME, f"--region={REGION}", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "networks", "subnets", "create", SUBNET_NAME, f"--network={NETWORK}", f"--region={REGION}", "--range=10.20.0.0/24", "--enable-private-ip-google-access", f"--project={PROJECT_ID}"])
if not resource_exists(["compute", "firewall-rules", "describe", "rainfall-internal", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "firewall-rules", "create", "rainfall-internal", f"--network={NETWORK}", "--source-ranges=10.20.0.0/24", "--allow=tcp,udp,icmp", f"--project={PROJECT_ID}"])
if not resource_exists(["iam", "service-accounts", "describe", VM_SERVICE_ACCOUNT, f"--project={PROJECT_ID}"]):
    run_gcloud(["iam", "service-accounts", "create", "rainfall-vm", "--display-name=Rainfall Dataproc VM", f"--project={PROJECT_ID}"])
# Dedicated VM identity; no service-account keys are created.
run_gcloud(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=serviceAccount:{VM_SERVICE_ACCOUNT}", "--role=roles/dataproc.worker", "--condition=None", "--format=none"])
print("Private subnet, internal-only firewall and dedicated VM identity prepared.")
if not resource_exists(["compute", "firewall-rules", "describe", "rainfall-iap-ssh", f"--project={PROJECT_ID}"]):
    run_gcloud(["compute", "firewall-rules", "create", "rainfall-iap-ssh", f"--network={NETWORK}", "--source-ranges=35.235.240.0/20", "--allow=tcp:22", "--target-tags=rainfall-cluster", f"--project={PROJECT_ID}"])
# Reuse the signed-in project owner for IAP; do not expose the human identity.
active_identity = subprocess.run(["gcloud", "config", "get-value", "account"], capture_output=True, text=True, check=True).stdout.strip()
run_gcloud(["projects", "add-iam-policy-binding", PROJECT_ID, f"--member=user:{active_identity}", "--role=roles/iap.tunnelResourceAccessor", "--condition=None", "--format=none"])
print("IAP SSH access prepared for the existing project owner; no public VM address.")


Private subnet, internal-only firewall and dedicated VM identity prepared.


IAP SSH access prepared for the existing project owner; no public VM address.


In [11]:
CREATE_BUCKET = True
UPLOAD_INPUTS_AND_CODE = True

if CREATE_BUCKET and not resource_exists(["storage", "buckets", "describe", f"gs://{BUCKET}"]):
    if not BUCKET:
        raise RuntimeError("Set a verified private bucket name first.")
    require_cost_gate(PLANNED_SESSION_COST_USD)
    run_gcloud([
        "storage", "buckets", "create", f"gs://{BUCKET}",
        f"--project={PROJECT_ID}",
        f"--location={REGION}",
        "--default-storage-class=STANDARD",
        "--uniform-bucket-level-access",
    ])
    print(f"Created private bucket gs://{BUCKET}")

run_gcloud(["storage", "buckets", "add-iam-policy-binding", f"gs://{BUCKET}", f"--member=serviceAccount:{VM_SERVICE_ACCOUNT}", "--role=roles/storage.objectAdmin", "--format=none"])

if UPLOAD_INPUTS_AND_CODE:
    if not BUCKET:
        raise RuntimeError("Set a verified private bucket name first.")
    require_cost_gate(PLANNED_SESSION_COST_USD)
    manifest_path = REPO_ROOT / "manifests" / "input_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    run_gcloud(["storage", "cp", str(manifest_path), f"gs://{BUCKET}/manifests/"])
    for record in manifest["files"]:
        year = record["year"]
        source = DATA_ROOT / "raw" / f"HistoricalRainfallacrossSingapore{year}.csv"
        if not source.is_file():
            raise FileNotFoundError(source)
        target_uri = f"gs://{BUCKET}/{record['relative_path']}"
        if resource_exists(["storage", "objects", "describe", target_uri]):
            object_info = run_gcloud(["storage", "objects", "describe", target_uri, "--format=json"], json_output=True)
            if object_info.get("custom_fields", object_info.get("metadata", {})).get("sha256") != record["sha256"]:
                raise RuntimeError("Existing object lacks the expected SHA-256 metadata; do not overwrite it.")
            print(f"Already uploaded: {target_uri}")
        else:
            run_gcloud(["storage", "cp", str(source), target_uri, f"--custom-metadata=sha256={record['sha256']}"])
            print(f"Uploaded {year}: {record['size_bytes']} bytes")
    for source in [
        REPO_ROOT / "scripts" / "run_quality_audit.py",
        REPO_ROOT / "scripts" / "build_manifest.py",
        REPO_ROOT / "notebooks" / "01_data_quality.ipynb",
    ]:
        run_gcloud(["storage", "cp", str(source), f"gs://{BUCKET}/code/"])
    print(f"Uploaded manifest, eight selected CSVs, and notebook code to gs://{BUCKET}/")
else:
    print("Bucket creation and uploads skipped; both are cost-bearing and require a refreshed cost gate.")

Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2017.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2018.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2019.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2020.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2021.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2022.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2023.csv


Already uploaded: gs://sg-rainfall-dsa5208-261011-data/raw/HistoricalRainfallacrossSingapore2024.csv


Uploaded manifest, eight selected CSVs, and notebook code to gs://sg-rainfall-dsa5208-261011-data/


In [12]:
# Bundle only explicit project code/configuration; exclude secrets and raw data.
import hashlib
import sys
import zipfile
sys.path.insert(0, str(REPO_ROOT / "src"))
from rainfall.config import load_config, config_hash
RUN_ID = "20261011-full-01"
artifact_dir = REPO_ROOT / "artifacts" / "execution"
artifact_dir.mkdir(parents=True, exist_ok=True)
bundle = artifact_dir / "source_bundle.zip"
files_to_bundle = list((REPO_ROOT / "src" / "rainfall").glob("*.py"))
files_to_bundle += list((REPO_ROOT / "scripts").glob("*.py"))
files_to_bundle += [REPO_ROOT / "config" / "experiment.json", REPO_ROOT / "manifests" / "input_manifest.json", REPO_ROOT / "requirements-cluster.txt"]
files_to_bundle += [path for path in (REPO_ROOT / "notebooks").glob("*.ipynb") if path.name[:2] in {"01", "02", "03", "04", "05"}]
wheel = REPO_ROOT / "artifacts" / "wheels" / "scienceplots-2.2.2-py3-none-any.whl"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(files_to_bundle):
        archive.write(path, str(path.relative_to(REPO_ROOT)))
    archive.write(wheel, f"wheels/{wheel.name}")
config = load_config(REPO_ROOT / "config" / "experiment.json")
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
runtime = {"project_id": PROJECT_ID, "bucket": BUCKET, "run_id": RUN_ID, "billing_account_id": BILLING_ACCOUNT_ID, "credit_id": ALLOWED_CREDIT_ID, "config_sha256": config_hash(config), "source_bundle_sha256": hashlib.sha256(bundle.read_bytes()).hexdigest(), "git_commit": commit}
runtime_path = artifact_dir / "runtime.json"
runtime_path.write_text(json.dumps(runtime, indent=2) + "\n")
require_cost_gate(PLANNED_SESSION_COST_USD)
run_gcloud(["storage", "cp", str(bundle), f"gs://{BUCKET}/code/source_bundle.zip"])
run_gcloud(["storage", "cp", str(runtime_path), f"gs://{BUCKET}/config/runtime.json"])
for path in files_to_bundle:
    if path.suffix == ".ipynb":
        run_gcloud(["storage", "cp", str(path), f"gs://{BUCKET}/notebooks/source/{path.name}"])
display(runtime)


{'project_id': 'sg-rainfall-dsa5208-261011',
 'bucket': 'sg-rainfall-dsa5208-261011-data',
 'run_id': '20261011-full-01',
 'billing_account_id': '01DAD8-AE5F1A-E3910A',
 'credit_id': '3LU63BPGY77JFDQQ',
 'config_sha256': 'ceee8ad94b7389bf33d81946006949eab04022b2fb83624d0eed444a6ca99cf5',
 'source_bundle_sha256': 'b9898907ace03b2d6ec7bff395b64c0c3229077365bb7c99780ee19cb5d89b0b',
 'git_commit': '745b0addba4faaa36d0eaeda4b547e8abadf4912'}

In [13]:
CREATE_CLUSTER = True
DATAPROC_IMAGE = "2.2.87-debian12"  # Pin an exact supported image after a no-cost compatibility review.
SUBNET = SUBNET_NAME
COST_ESTIMATE_USD = PLANNED_SESSION_COST_USD

if CREATE_CLUSTER and not resource_exists(["dataproc", "clusters", "describe", CLUSTER, f"--project={PROJECT_ID}", f"--region={REGION}"]):
    if not BUCKET or not SUBNET or not VM_SERVICE_ACCOUNT or not DATAPROC_IMAGE:
        raise RuntimeError("Set bucket, subnet, service account, and exact image first.")
    require_cost_gate(COST_ESTIMATE_USD)
    run_gcloud([
        "dataproc", "clusters", "create", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}",
        f"--image-version={DATAPROC_IMAGE}",
        f"--subnet={SUBNET}", "--no-address",
        f"--service-account={VM_SERVICE_ACCOUNT}", "--scopes=cloud-platform",
        "--master-machine-type=e2-standard-4",
        "--master-boot-disk-type=pd-balanced", "--master-boot-disk-size=100GB",
        "--num-workers=2", "--worker-machine-type=e2-standard-4",
        "--worker-boot-disk-type=pd-balanced", "--worker-boot-disk-size=75GB",
        f"--bucket={BUCKET}", f"--temp-bucket={BUCKET}",
        "--tags=rainfall-cluster", "--async", "--enable-component-gateway", "--optional-components=JUPYTER",
        f"--properties=dataproc:jupyter.notebook.gcs.dir=gs://{BUCKET}/notebooks/,spark:spark.eventLog.enabled=true,spark:spark.eventLog.dir=gs://{BUCKET}/spark-history/",
        "--delete-max-idle=45m", "--delete-max-age=3h",
    ])
    print(f"Requested Dataproc cluster {CLUSTER} in {REGION}.")
else:
    print("Cluster already exists or creation was skipped; inspect its state before execution.")

Requested Dataproc cluster dsa5208-rainfall in asia-southeast1.


In [14]:
cluster_info = run_gcloud(["dataproc", "clusters", "describe", CLUSTER, f"--project={PROJECT_ID}", f"--region={REGION}", "--format=json"], json_output=True)
display({"cluster": CLUSTER, "state": cluster_info["status"]["state"], "master": cluster_info["config"]["masterConfig"]["instanceNames"], "component_urls": cluster_info["config"].get("endpointConfig", {}).get("httpPorts", {}), "auto_delete": cluster_info["config"].get("lifecycleConfig")})


{'cluster': 'dsa5208-rainfall',
 'state': 'CREATING',
 'master': ['dsa5208-rainfall-m'],
 'component_urls': {'HDFS NameNode': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/hdfs/dfshealth.html',
  'Jupyter': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/gateway/default/jupyter/',
  'JupyterLab': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/gateway/default/jupyter/lab/',
  'MapReduce Job History': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/jobhistory/',
  'Spark History Server': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/sparkhistory/',
  'Tez': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/apphistory/tez-ui/',
  'YARN Application Timeline': 'https://mqso6q4ntbazvdhjpu5vl5nmlm-dot-asia-southeast1.dataproc.googleusercontent.com/apphistory/'

In [15]:
DELETE_CLUSTER = False

if DELETE_CLUSTER:
    run_gcloud([
        "dataproc", "clusters", "delete", CLUSTER,
        f"--project={PROJECT_ID}", f"--region={REGION}", "--quiet",
    ])
    remaining = run_gcloud([
        "dataproc", "clusters", "list",
        f"--project={PROJECT_ID}", f"--region={REGION}",
        "--format=json",
    ], json_output=True)
    display({"remaining_cluster_names": [item["clusterName"] for item in remaining]})
else:
    print("Cluster deletion skipped. Set DELETE_CLUSTER=True after outputs and notebooks are persisted.")

Cluster deletion skipped. Set DELETE_CLUSTER=True after outputs and notebooks are persisted.


## Spark Web UI evidence

After each Spark action in `01_data_quality.ipynb` or a later analysis notebook finishes, open the corresponding **Spark Web UI**. Review the Jobs summary and every Job detail page, then save screenshots under `report/evidence/spark-ui/<RUN_ID>/`. Use Spark History Server for a completed application. Do not substitute Dataproc Jobs or YARN screenshots.